# Generation server — Feature 004, Phase 0 provisioning

This notebook **provisions** the generation server the Phase 0 first-token benchmark
measures. It is Phase 0 work, not Phase 4 work: Phase 4 reuses this artefact and this
contract rather than creating either again (FR-035o).

Run the cells in order. Each one refuses to continue when its prerequisite is absent,
rather than carrying a half-provisioned server forward into a measurement.

## What must never travel to this session

This runs on someone else's computer, reached over a public tunnel. The trust boundary
is therefore narrow and worth stating in full — it is easier to keep than to restore:

* **No browser JWTs, signing keys, refresh tokens, or session cookies.** The existing API
  remains the only verifier of user session tokens. This server never sees one.
* **No access-context objects, ACL records, permission fingerprints, or excluded-source
  counts.** Authorization is decided before retrieval, locally, and its inputs stay there.
* **No unauthorized chunks.** Only passages the asker is already entitled to read are sent,
  and they are sent as text with no attached permission metadata.
* **No real enterprise or personal data.** This profile is permitted for **the project's
  synthetic corpus only** (FR-011h). Real data requires an approved private or self-hosted
  endpoint, which this is not.

## Secrets

`NGROK_AUTHTOKEN` is read from **Colab Secrets only** — never pasted into a cell, never
committed, never printed. The service token this notebook mints is printed once, for you
to copy into an **ignored** `.env`. Neither value belongs in the repository or in a log.

## Licence

Qwen2.5-3B-Instruct is the **Qwen RESEARCH LICENSE**: non-commercial, research and
evaluation only. `docs/models.md` records the clauses. Weights stay in this ephemeral
session and are never committed or baked into an image.

## 1 — Verify the GPU is the declared T4

The T4 is the **latency reference class**, not a floor (FR-035c, FR-043a). A faster
allocation is refused for the same reason a slower one is: the resulting figure would
describe hardware the threshold was never defined against.

In [ ]:
import subprocess
import sys

gpu = subprocess.run(
    ["nvidia-smi", "--query-gpu=name,memory.total", "--format=csv,noheader"],
    capture_output=True, text=True, check=False,
).stdout.strip()

print("allocated:", gpu or "<none>")

if not gpu:
    sys.exit("INVALID: no GPU allocated. Runtime > Change runtime type > T4 GPU.")
if "T4" not in gpu:
    sys.exit(
        f"INVALID: allocated {gpu!r}, which is not the declared T4 reference class.\n"
        "A different allocation - faster or slower - produces a first-token figure that\n"
        "describes hardware the threshold was never defined against. Recycle the runtime\n"
        "until a T4 is allocated, or record the run INVALID rather than as a pass."
    )

GPU_NAME = gpu.split(",")[0].strip()
print("verified T4:", GPU_NAME)

## 2 — Pinned runtime

Exact versions. An unpinned runtime changes generation behaviour between sessions, and a
quality figure attributed to "llama.cpp, some build" is not attributable at all (FR-011b).

In [ ]:
!pip install --quiet llama-cpp-python==0.3.5 fastapi==0.115.6 uvicorn==0.34.0 \
    pyngrok==7.2.3 huggingface-hub==0.27.1

import llama_cpp

RUNTIME_IDENTITY = f"llama-cpp-python/{llama_cpp.__version__}"
QUANTIZATION = "Q4_K_M"
print(RUNTIME_IDENTITY, QUANTIZATION)

## 3 — Pinned weights, revision and checksum verified

Acquisition is a **provisioning** activity, and it verifies what it acquired. An unverified
download is a guess about what will run (FR-011f). The expected values come from
`docs/models.md`; if they disagree, the mismatch is the finding.

In [ ]:
import hashlib
import sys

from huggingface_hub import hf_hub_download

REPOSITORY = "Qwen/Qwen2.5-3B-Instruct-GGUF"
REVISION = "7dabda4d13d513e3e842b20f0d435c732f172cbe"
FILENAME = "qwen2.5-3b-instruct-q4_k_m.gguf"
EXPECTED_SHA256 = "626b4a6678b86442240e33df819e00132d3ba7dddfe1cdc4fbb18e0a9615c62d"

WEIGHTS = hf_hub_download(
    repo_id=REPOSITORY, filename=FILENAME, revision=REVISION, local_dir="/content/models"
)

digest = hashlib.sha256()
with open(WEIGHTS, "rb") as handle:
    for block in iter(lambda: handle.read(1024 * 1024), b""):
        digest.update(block)
WEIGHTS_SHA256 = digest.hexdigest()

if WEIGHTS_SHA256 != EXPECTED_SHA256:
    sys.exit(
        f"INVALID: {FILENAME} hashed {WEIGHTS_SHA256}, expected {EXPECTED_SHA256}.\n"
        "These are different weights. Any figure measured against them would be\n"
        "attributed to a model nobody pinned."
    )
print("weights verified:", WEIGHTS_SHA256)

## 4 — Mint the service token

The tunnel URL is public the moment it exists. The token is what makes the endpoint
*authenticated* rather than merely obscure (FR-028g).

Printed once. Copy it into an **ignored** `.env` — never into the repository, never into a
log, never into a commit.

In [ ]:
import secrets

SERVICE_TOKEN = secrets.token_urlsafe(32)
print("GENERATION_SERVICE_TOKEN=" + SERVICE_TOKEN)

## 5 — The server

Three things the benchmark and Phase 4 both depend on:

* **`/health`** — reports readiness plus the identity of what is loaded, so a caller can
  tell *this* server apart from a different one at the same URL (FR-028k).
* **Streaming** — the first `token` event is what the first-token benchmark clocks. It is
  emitted as soon as the model produces it, never buffered.
* **Cancellation** — a disconnected client stops generation. Without this a cancelled turn
  keeps consuming the GPU, and "stop" means "stop showing me" rather than "stop" (FR-025a).

Every request is authenticated. An unauthenticated request is refused before the model is
touched, so an unauthorized caller cannot consume the GPU either.

In [ ]:
import asyncio
import json
import secrets
import time

from fastapi import FastAPI, Header, HTTPException, Request
from fastapi.responses import StreamingResponse
from llama_cpp import Llama

llm = Llama(
    model_path=WEIGHTS,
    n_gpu_layers=-1,      # the whole model on the T4; partial offload is a different machine
    n_ctx=4096,           # five 400-token passages, the question, and the instruction
    seed=0,               # deterministic where the runtime supports it (FR-011a)
    verbose=False,
)

app = FastAPI()


def _authenticate(authorization: str | None) -> None:
    expected = f"Bearer {SERVICE_TOKEN}"
    # `compare_digest` so a wrong token cannot be recovered by timing the comparison.
    if not authorization or not secrets.compare_digest(authorization, expected):
        raise HTTPException(status_code=401, detail="unauthenticated")


@app.get("/health")
def health() -> dict:
    """Unauthenticated on purpose: it reveals no corpus content and carries no secret."""
    return {
        "status": "ok",
        "model_repository": REPOSITORY,
        "model_revision": REVISION,
        "weights_sha256": WEIGHTS_SHA256,
        "quantization": QUANTIZATION,
        "runtime_identity": RUNTIME_IDENTITY,
        "gpu_name": GPU_NAME,
        "context_length": 4096,
    }


@app.post("/generate")
async def generate(request: Request, authorization: str | None = Header(default=None)):
    _authenticate(authorization)
    body = await request.json()

    async def stream():
        started = time.perf_counter()
        first = True
        completion = llm.create_completion(
            prompt=body["prompt"],
            max_tokens=int(body.get("max_tokens", 512)),
            temperature=float(body.get("temperature", 0.0)),
            top_p=float(body.get("top_p", 1.0)),
            stream=True,
        )
        try:
            for piece in completion:
                # An unobserved disconnect is still a cancellation (FR-025b). Checking it
                # every token is what makes the GPU stop rather than finish into a void.
                if await request.is_disconnected():
                    yield f"event: cancelled\ndata: {json.dumps({'reason': 'client_disconnect'})}\n\n"
                    return
                text = piece["choices"][0]["text"]
                if not text:
                    continue
                if first:
                    payload = {"text": text, "first_token_ms": (time.perf_counter() - started) * 1000}
                    first = False
                else:
                    payload = {"text": text}
                yield f"event: token\ndata: {json.dumps(payload)}\n\n"
                await asyncio.sleep(0)
            yield "event: done\ndata: {}\n\n"
        finally:
            completion.close()

    return StreamingResponse(stream(), media_type="text/event-stream")

print("server defined")

## 6 — The authenticated HTTPS tunnel

`NGROK_AUTHTOKEN` comes from **Colab Secrets** (the key icon in the sidebar). It is never
pasted into a cell, because a pasted secret is saved in the notebook's output and travels
with any copy of it.

In [ ]:
import sys
import threading

import nest_asyncio
import uvicorn
from google.colab import userdata
from pyngrok import ngrok

try:
    NGROK_AUTHTOKEN = userdata.get("NGROK_AUTHTOKEN")
except Exception as error:
    sys.exit(
        f"NGROK_AUTHTOKEN is not available from Colab Secrets ({error}).\n"
        "Add it under the key icon in the sidebar. Do not paste it into a cell:\n"
        "cell input and output are saved with the notebook."
    )

ngrok.set_auth_token(NGROK_AUTHTOKEN)
nest_asyncio.apply()

tunnel = ngrok.connect(8000, "http", bind_tls=True)   # bind_tls: HTTPS only, no plaintext
GENERATION_URL = tunnel.public_url

if not GENERATION_URL.startswith("https://"):
    sys.exit(f"INVALID: tunnel is not HTTPS: {GENERATION_URL}")

threading.Thread(
    target=lambda: uvicorn.run(app, host="0.0.0.0", port=8000, log_level="warning"),
    daemon=True,
).start()

print("Copy these two lines into your ignored .env — never commit them:\n")
print("GENERATION_URL=" + GENERATION_URL)
print("GENERATION_SERVICE_TOKEN=" + SERVICE_TOKEN)

## 7 — Self-check: the seven prerequisites

The same seven `benchmarks/phase0/server_provisioning.py` verifies (FR-035o). Running them
here means the notebook reports its own readiness, rather than the benchmark discovering a
gap after someone has waited for a session.

In [ ]:
import urllib.request

observed = {
    "weights_revision": REVISION,
    "weights_checksum": WEIGHTS_SHA256,
    "endpoint_url": GENERATION_URL,
    "service_token": SERVICE_TOKEN,
    "gpu_name": GPU_NAME,
    "runtime_identity": RUNTIME_IDENTITY,
    "quantization": QUANTIZATION,
    "health_ok": False,
    "streams_first_token": False,
}

with urllib.request.urlopen(GENERATION_URL + "/health", timeout=30) as response:
    observed["health_ok"] = response.status == 200

probe = urllib.request.Request(
    GENERATION_URL + "/generate",
    data=json.dumps({"prompt": "Reply with the single word: ready.", "max_tokens": 4}).encode(),
    headers={"Authorization": f"Bearer {SERVICE_TOKEN}", "Content-Type": "application/json"},
)
with urllib.request.urlopen(probe, timeout=120) as response:
    for raw in response:
        if raw.startswith(b"event: token"):
            observed["streams_first_token"] = True
            break

for key, value in observed.items():
    shown = "<set>" if key == "service_token" else value
    print(f"  {key}: {shown}")

print(
    "\nPaste these into the benchmark host and run `make benchmark-phase0`."
    "\nThe verifier records NOT RUN or INVALID if any prerequisite is absent — never a pass."
)